# Exercícios 01 — Tradução de DNA

Bioinformática — Fundamentos  
**Arquivo de sequências:** `sequence.txt`  
**Tabela de códons:** `codons.csv`

---

## Visão Geral

Este notebook implementa o processo de **tradução de sequências de DNA em proteínas**, seguindo o fluxo central da biologia molecular:

```
DNA (A, T, G, C)  →  Códon (3 nucleotídeos)  →  Aminoácido  →  Proteína
```

### Conceitos abordados

| Conceito | Descrição |
|---|---|
| **Nucleotídeo** | Unidade básica do DNA: A (adenina), T (timina), G (guanina), C (citosina) |
| **Códon** | Tripla de nucleotídeos que codifica um aminoácido |
| **Aminoácido** | Monômero das proteínas, representado por uma letra (ex.: `M` = metionina) |
| **Códon de início** | `ATG` — sinaliza o início da tradução (codifica metionina, `M`) |
| **Códon de parada** | `TAA`, `TAG` ou `TGA` — sinaliza o fim da tradução |
| **Reading frame** | Posição de início da leitura (0, 1 ou 2); altera completamente o resultado |
| **ORF** | *Open Reading Frame* — trecho lido do códon de início até o de parada |


---
## 1. Carregamento dos Dados

In [ ]:
# Lê as sequências de DNA do arquivo de texto.
# Cada linha contém uma sequência distinta; strip() remove espaços e quebras de linha.
with open('sequence.txt', 'r') as f:
    sequences = [linha.strip() for linha in f.readlines()]

print(f"{len(sequences)} sequências carregadas:")
for i, seq in enumerate(sequences):
    print(f"  [{i}] {seq}")

---
## 2. Tabela de Códons

O arquivo `codons.csv` mapeia cada tripla de nucleotídeos ao aminoácido correspondente.  
Códons especiais (`TAA`, `TAG`, `TGA`) são mapeados para `STOP`, indicando fim da proteína.

In [30]:
import pandas as pd

# Carrega a tabela e cria um dicionário códon → aminoácido para consultas rápidas
df_codons = pd.read_csv('codons.csv')
CODON_TO_AA = dict(zip(df_codons["Codon"], df_codons["AA"]))

print(f"Total de códons na tabela: {len(CODON_TO_AA)}")
stop_codons = [c for c, aa in CODON_TO_AA.items() if aa == 'STOP']
print(f"Códons de parada: {stop_codons}")
print("\nPrimeiros 10 mapeamentos:")
for codon, aa in list(CODON_TO_AA.items())[:10]:
    print(f"  {codon} → {aa}")

Total de códons na tabela: 64
Códons de parada: ['TAA', 'TAG', 'TGA']

Primeiros 10 mapeamentos:
  TTT → F
  TTC → F
  TTA → L
  TTG → L
  TCT → S
  TCC → S
  TCA → S
  TCG → S
  TAT → Y
  TAC → Y


---
## 3. Função de Tradução

A função `translate` percorre a sequência de DNA a partir de uma posição (`start_frame`),  
lendo códons de 3 em 3 nucleotídeos, e retorna a cadeia de aminoácidos correspondente.

A leitura para quando:
- Encontra um **códon de parada** (`TAA`, `TAG`, `TGA`)
- Não há mais nucleotídeos suficientes para formar um códon completo

In [31]:
def translate(seq: str, start_frame: int) -> str:
    """
    Traduz uma sequência de DNA em uma cadeia de aminoácidos.

    Parâmetros
    ----------
    seq : str
        Sequência de DNA contendo apenas os caracteres A, T, G, C.
    start_frame : int
        Posição de início da leitura (0, 1 ou 2).

    Retorna
    -------
    str
        Sequência de aminoácidos (string de letras de uma letra) ou
        string vazia se nenhum aminoácido for produzido antes do STOP.
    """
    aminoacids = []
    for i in range(start_frame, len(seq) - 2, 3):
        codon = seq[i:i+3]
        aa = CODON_TO_AA.get(codon)
        if aa is None:          # Códon desconhecido — pula
            continue
        if aa == "STOP":        # Fim da tradução
            break
        aminoacids.append(aa)
    return "".join(aminoacids)


# Exemplo de uso com a primeira sequência nos três reading frames
seq_exemplo = sequences[0]
print(f"Sequência: {seq_exemplo}")
for frame in range(3):
    print(f"  Frame {frame}: {translate(seq_exemplo, frame)}")

Sequência: ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
  Frame 0: MAIVMGR
  Frame 1: WPL
  Frame 2: GHCNGPLKGCPI


---
## 4. Análise dos Reading Frames

Para cada sequência, calculamos a tradução nos três reading frames possíveis (0, 1, 2),  
registrando também o comprimento de cada proteína gerada e identificando qual frame  
produz a proteína mais longa e qual produz a mais curta.

In [32]:
start_frames = [0, 1, 2]
df_analysis = []

for seq in sequences:
    if not seq:
        continue

    # Traduz nos três frames e calcula tamanhos
    traducoes = {f: translate(seq, f) for f in start_frames}
    tamanhos  = {f: len(t) for f, t in traducoes.items()}

    df_analysis.append({
        "sequence":       seq,
        "start_frame_0":  traducoes[0],
        "start_frame_1":  traducoes[1],
        "start_frame_2":  traducoes[2],
        "len_frame_0":    tamanhos[0],
        "len_frame_1":    tamanhos[1],
        "len_frame_2":    tamanhos[2],
        "longest_frame":  max(tamanhos, key=tamanhos.get),
        "shortest_frame": min(tamanhos, key=tamanhos.get),
    })

df = pd.DataFrame(df_analysis)
df

,sequence,start_frame_0,start_frame_1,start_frame_2,len_frame_0,len_frame_1,len_frame_2,longest_frame,shortest_frame
0,ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG,MAIVMGR,WPL,GHCNGPLKGCPI,7,3,12,2,1
1,ATGAAATTTGGGCCCTAA,MKFGP,,EIWAL,5,0,5,0,1
2,ATGAAAATGAAATAG,MKMK,,ENEI,4,0,4,0,1
3,ATGTTTAAATGCCCTAG,MFKCP,CLNAL,V,5,5,1,0,2
4,ATGAAAAAAATAGATGCCCCCTAA,MKKIDAP,,EKNRCPL,7,0,7,0,1
5,ATGAAAGGGAAATTTTAG,MKGKF,,EREIL,5,0,5,0,1
6,ATGCCCCCCCCCTAA,MPPP,CPPP,APPL,4,4,4,0,0
7,ATGAAAATAAAATAA,MKIK,,ENKI,4,0,4,0,1
8,ATGTTTTTTTTTTTTTTAG,MFFFFL,CFFFF,VFFFF,6,5,5,0,1
9,ATGGCCATTTGATGTCCAATGGCCGCTGCCGCTGAATAG,MAI,WPFDVQWPLPLN,GHLMSNGRCR,3,12,10,1,0


---
## 5. Identificação de ORFs (*Open Reading Frames*)

Uma ORF começa sempre em um **códon ATG** (metionina) e vai até o próximo códon de parada.  
Uma mesma sequência pode ter múltiplas ORFs em posições diferentes.

O pipeline abaixo:
1. Encontra todas as posições de `ATG` na sequência
2. Traduz a partir de cada posição encontrada
3. Seleciona a ORF mais longa como candidata à proteína de interesse

In [33]:
def find_start_codons(seq: str) -> list[int]:
    """
    Retorna as posições (índices) de todos os códons de início (ATG) na sequência.

    Parâmetros
    ----------
    seq : str
        Sequência de DNA.

    Retorna
    -------
    list[int]
        Lista de índices onde 'ATG' foi encontrado.
    """
    return [i for i in range(len(seq) - 2) if seq[i:i+3] == "ATG"]


# Demonstração com a primeira sequência
seq_demo = sequences[0]
posicoes = find_start_codons(seq_demo)
print(f"Sequência: {seq_demo}")
print(f"Posições de ATG: {posicoes}")
print("\nTraduções a partir de cada ATG:")
for pos in posicoes:
    print(f"  Posição {pos:>2}: {translate(seq_demo, pos)}")

Sequência: ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
Posições de ATG: [0, 12]

Traduções a partir de cada ATG:
  Posição  0: MAIVMGR
  Posição 12: MGR


In [34]:
def find_orfs(seq: str) -> list[str]:
    """
    Encontra todas as ORFs de uma sequência, traduzindo a partir de cada ATG.

    Parâmetros
    ----------
    seq : str
        Sequência de DNA.

    Retorna
    -------
    list[str]
        Lista de sequências de aminoácidos (uma por ATG encontrado).
    """
    return [translate(seq, pos) for pos in find_start_codons(seq)]


def select_longest_orf(orfs: list[str]) -> str:
    """
    Retorna a ORF mais longa de uma lista de ORFs.

    Parâmetros
    ----------
    orfs : list[str]
        Lista de sequências de aminoácidos.

    Retorna
    -------
    str
        A sequência de aminoácidos com maior comprimento.
    """
    return max(orfs, key=len)


# Demonstração: encontra e seleciona a ORF mais longa da primeira sequência
orfs = find_orfs(sequences[0])
print(f"ORFs encontradas: {orfs}")
print(f"ORF mais longa:   {select_longest_orf(orfs)}")

ORFs encontradas: ['MAIVMGR', 'MGR']
ORF mais longa:   MAIVMGR


---
## 6. Pipeline Completo — ORF Mais Longa por Sequência

Aplicamos o pipeline completo a todas as sequências do arquivo,  
exibindo para cada uma: as ORFs encontradas e a candidata a proteína principal.

In [35]:
resultados = []

for i, seq in enumerate(sequences):
    orfs = find_orfs(seq)
    if not orfs:
        longest = ""
    else:
        longest = select_longest_orf(orfs)

    resultados.append({
        "#":            i,
        "sequence":     seq,
        "num_orfs":     len(orfs),
        "all_orfs":     " | ".join(orfs),
        "longest_orf":  longest,
        "protein_len":  len(longest),
    })

df_orfs = pd.DataFrame(resultados)
df_orfs

,#,sequence,num_orfs,all_orfs,longest_orf,protein_len
0,0,ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG,2,MAIVMGR | MGR,MAIVMGR,7
1,1,ATGAAATTTGGGCCCTAA,1,MKFGP,MKFGP,5
2,2,ATGAAAATGAAATAG,2,MKMK | MK,MKMK,4
3,3,ATGTTTAAATGCCCTAG,2,MFKCP | MP,MFKCP,5
4,4,ATGAAAAAAATAGATGCCCCCTAA,2,MKKIDAP | MPP,MKKIDAP,7
5,5,ATGAAAGGGAAATTTTAG,1,MKGKF,MKGKF,5
6,6,ATGCCCCCCCCCTAA,1,MPPP,MPPP,4
7,7,ATGAAAATAAAATAA,1,MKIK,MKIK,4
8,8,ATGTTTTTTTTTTTTTTAG,1,MFFFFL,MFFFFL,6
9,9,ATGGCCATTTGATGTCCAATGGCCGCTGCCGCTGAATAG,3,MAI | MSNGRCR | MAAAAE,MSNGRCR,7
